In [ ]:

# Advanced RAG - Data Ingestion Pipeline
# Page-wise Document Processing with Filename-based Metadata Extraction
from dotenv import load_dotenv
load_dotenv()

import hashlib  #creare hash (es: dedup documenti)
from pathlib import Path  #gestione filesystem moderno

from langchain_chroma import Chroma  #vector database locale basato su ChromaDB
from langchain_ollama import OllamaEmbeddings  #x wrapper x LLMembeddings
from langchain_core.documents import Document   #x struttura standard langchain  Document(page_content="text",metadata={...})

from docling.document_converter import DocumentConverter  #x parsing documenti complessi PDF/DOCX testo pulito layout-aware parsing

In [ ]:
DATA_DIR = "data"  
CHROMA_DIR = "./chroma_financial_db"  #dove salvare db
COLLECTION_NAME = "financial_docs"
EMBEDDING_MODEL = 'nomic-embed-text'
BASE_URL = 'http://localhost:11434'

embeddings= OllamaEmbeddings(
    model=EMBEDDING_MODEL, 
    base_url=BASE_URL, 
    num_ctx=8192)

vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,  #usa questo embedding model (l'ho creato qua sopra)
    persist_directory=CHROMA_DIR
)

In [ ]:
"""
    Extract metadata from filename.
    
    Expected format: {company} {doc_type} {quarter} {year}.pdf
    Examples:
    - amazon 10-k 2024.pdf

    Returns:
        dict with company_name, doc_type, fiscal_year, fiscal_quarter
    """

def extract_metadata_from_filename(filename: str) -> dict:
    name = filename.replace('.pdf', '')   #rimuove extension .pdf
    parts = name.split()  #separa per spazi e.g. "amazon 10-k 2024"->["amazon", "10-k", "2024"]
    metadata = {}  #dictionary
    if len(parts) == 4:
        metadata['fiscal_quarter'] = parts[2]  #setta nuova coppia key-value nel dictionary
        metadata['fiscal_year'] = int(parts[3])
    else:
        metadata['fiscal_quarter'] = None
        metadata['fiscal_year'] = int(parts[2])
    metadata['company_name'] = parts[0]
    metadata['doc_type'] = parts[1]
    return metadata

extract_metadata_from_filename('amazon 10-k 2023.pdf')
extract_metadata_from_filename('amazon 10-q q1 2024.pdf')  #qua ci saranno 4 parts

In [ ]:
def extract_pdf_pages(pdf_path):
    converter = DocumentConverter()   #PDF→testo converter intelligente, legge PDF e capisce struttura (titoli, tabelle, paragrafi), è proprio parsing semantico!
    result = converter.convert(pdf_path)  #legge il pdf->trasforma in struttura documentale, ora il result contiene testo-struttura-pagine-metadatainterni
    page_break = "<!-- page break -->"
    markdown_text = result.document.export_to_markdown(page_break_placeholder=page_break)  #converti in markdown, ogni page sara separata da <!-- page break -->
    pages = markdown_text.split(page_break)  #suddividi in list ["pagina 1...","pagina 2...",...] basandoti su page_break!!
    return pages

pages = extract_pdf_pages('data/amazon/amazon 10-q q1 2024.pdf')
len(pages)  #quante pagine

# """Compute SHA-256 hash of file content."""

def compute_file_hash(file_path: str) -> str:  #x generare ID unico del file
    sha256_hash = hashlib.sha256()  #crei algoritmo hash SHA-256
    with open(file_path, "rb") as f:  #apri file in binary mode, xk hash lavora su byte
        for byte_block in iter(lambda: f.read(4096), b""):  #leggi il file a blocchi da 4KB(4096bytes), iter(lambda: ..., b"") equivale a fare e.g.
            #while True:
                #chunk = f.read(4096)
                #if chunk == b"":
                    #break
                # ...
            sha256_hash.update(byte_block)  #aggiunge quel blocco al calcolo hash
        #in questo modo l'hash viene creato progressivamente, BETTER X RAM!!
    return sha256_hash.hexdigest()  #converte in stranga esadecimale leggibile e.g."a3f9c2..."

compute_file_hash('data/amazon/amazon 10-q q1 2024.pdf')
compute_file_hash('data/amazon/amazon 10-q q1 2024 copy.pdf')

In [ ]:
# track the processed files
existing_docs = vector_store.get(
    where={"file_hash": {"$ne": ""}},  #
    include=['metadatas'])  
    #"dammi i metadata dei documenti che hanno file_hash", prendi tutti i documenti dove file_hash !="", e anche i metadatas
processed_hashes = [
    m.get('file_hash') 
    for m in existing_docs['metadatas'] 
    if m.get('file_hash')]  
    #fai lista di hash già presenti, prendi metadata estrai file_hash ignori quelli null
processed_hashes = set(processed_hashes)  #TRASFORMI IN SET (è una struttura dati come List/LinkedList/ect)(cosi hai lookup O(1)!)
processed_hashes

In [ ]:
def ingest_docs_in_vectordb(pdf_path):
    print(f"Processing: {pdf_path.name}")
    file_hash = compute_file_hash(pdf_path)  #crei id univoko del file
    if file_hash in processed_hashes:   #iteri con condizione
        print(f"[SKIP] already processed: {pdf_path}")
        return 
    pages = extract_pdf_pages(pdf_path)  #
    file_metadata = extract_metadata_from_filename(pdf_path.name)
    processed_pages = []
    for page_num, page_text in enumerate(pages, start=1):  #1-index based
        metadata_dict = file_metadata.copy()  #clone
        metadata_dict['page'] = page_num
        metadata_dict['file_hash'] = file_hash
        metadata_dict['source_file'] = pdf_path.name
        doc = Document(page_content=page_text, metadata=metadata_dict)  #crea obj oggetto langchain Document
        processed_pages.append(doc)
    vector_store.add_documents(documents=processed_pages)  #embedding auto e carica su chroma db

In [ ]:

data_path = Path(DATA_DIR)
pdf_files = list(data_path.rglob("*.pdf"))  #trova tutti i pdf anche nelle sottocartelle
pdf_files[:3]  #show i primi 3

In [ ]:

for pdf_path in pdf_files:
    ingest_docs_in_vectordb(pdf_path)

vector_store._collection.count()
vector_store.get(where={"company_name": "amazon"}, limit=2)
results = vector_store.search("What is Tesla's revenue for Q1 2024", search_type="similarity") #search_type="similarity" trova chunk piu simili
results